<a href="https://colab.research.google.com/github/amiralito/Tiberius_colab/blob/main/Tiberius_gene_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tiberius — batch gene prediction + BUSCO on Google Colab

[Tiberius](https://github.com/Gaius-Augustus/Tiberius) is a deep-learning gene finder. It predicts gene structures from a **genome FASTA only** and writes **GTF/GFF3** (+ optional CDS/protein FASTA). This notebook runs Tiberius over **one or many genomes**, gzips the outputs, then scores completeness with **BUSCO** and compiles the results into a table.

**How to use:** set a GPU runtime (*Runtime > Change runtime type > GPU*; a free T4 is enough), then fill the form fields and run each cell top to bottom. **The code is hidden** — to edit a cell, open its **menu (vertical dots) > Show code**.

Mount Drive in cell 4 to read genomes from a folder and keep results across sessions. Outputs are named after each genome (`<genome>_gff.gff3.gz`, `<genome>_cds.fasta.gz`, `<genome>_protein.fasta.gz`) with gene/transcript IDs prefixed `tiberius_`.

In [ ]:
#@title 1 · Check runtime — GPU & Python { display-mode: "form" }
import sys
print("Python:", sys.version.split()[0])
assert sys.version_info >= (3, 12), "Tiberius needs Python >= 3.12 - use the default Colab runtime."
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU - Runtime > Change runtime type > GPU."

In [ ]:
#@title 2 · Install Tiberius { display-mode: "form" }
#@markdown Source install (no Singularity). Keeps Colab's GPU TensorFlow 2.19 (Tiberius pins `<2.21`). ~2 min.
%cd /content
![ -d Tiberius ] || git clone --depth 1 https://github.com/Gaius-Augustus/Tiberius
%cd /content/Tiberius
!pip install -q .[from_source]
print("Done. If the GPU is NOT listed in cell 3, do Runtime > Restart session, then re-run cells 1-3.")

In [ ]:
#@title 3 · Verify GPU is visible to TensorFlow { display-mode: "form" }
import importlib.metadata as m, tensorflow as tf
print("tiberius   :", m.version("tiberius"))
print("tensorflow :", tf.__version__)
gpus = tf.config.list_physical_devices("GPU"); print("GPUs       :", gpus)
assert gpus, "TF can't see the GPU. Runtime > Restart session, then re-run cells 1-3."
!python tiberius.py --list_cfg

In [ ]:
#@title 4 · Storage — mount Drive & set output folder { display-mode: "form" }
#@markdown Mount Google Drive so inputs/outputs persist across sessions. All results are written to **WORKDIR**.
MOUNT_DRIVE = True  #@param {type:"boolean"}
WORKDIR = "/content/drive/MyDrive/tiberius_runs"  #@param {type:"string"}

import os
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
elif WORKDIR.startswith("/content/drive"):
    WORKDIR = "/content/work"
os.makedirs(WORKDIR, exist_ok=True)
print("Working directory:", WORKDIR)

In [ ]:
#@title 5 · Prediction parameters & genome input { display-mode: "form" }
MODEL_CFG = "angiosperms"  #@param ["angiosperms", "vertebrates", "fungi", "insecta", "diatoms", "chlorophyta", "mammalia_softmasking_v2", "mammalia_nosofttmasking_v2", "mammalia_clamsa_v2", "model_cfg/superseded/monocotyledonae.yaml", "model_cfg/superseded/eudicotyledons.yaml", "model_cfg/superseded/angiosperms_softmasking.yaml", "model_cfg/superseded/diatoms_softmasking.yaml", "model_cfg/superseded/insecta_softmasking.yaml", "model_cfg/superseded/lepidoptera.yaml", "model_cfg/superseded/mucoromycota.yaml", "model_cfg/superseded/saccharomycota.yaml", "model_cfg/superseded/sordariomycota.yaml"] {allow-input: true}
#@markdown Top-level models are **unmasked**. The `superseded/` models are clade-specific and **softmasking-aware** (use the lowercase repeat mask) — useful for repeat-rich genomes, e.g. `monocotyledonae` for maize.
ID_PREFIX = "tiberius_"  #@param {type:"string"}
HARDMASK_REPEATS = False  #@param {type:"boolean"}
#@markdown `HARDMASK_REPEATS`: convert softmasked (lowercase) bases to `N` before prediction. Helps **unmasked** models on repeat-rich genomes; automatically skipped if a softmasking model is selected.
BATCH_SIZE = "auto"  #@param ["auto", "2", "4", "8", "16", "32"] {allow-input: true}
WANT_CODING = True  #@param {type:"boolean"}
WANT_PROTEIN = True  #@param {type:"boolean"}
#@markdown ---
#@markdown **Genome input** — `drive_dir` runs on **every** FASTA found recursively under the folder; the others take a single genome.
GENOME_SOURCE = "drive_dir"  #@param ["drive_dir", "upload", "url", "drive_file"]
GENOME_DIR = "/content/drive/MyDrive/genomes"  #@param {type:"string"}
GENOME_PATH = "/content/drive/MyDrive/genome.fasta"  #@param {type:"string"}
GENOME_URL = ""  #@param {type:"string"}
print("model", MODEL_CFG, "| id_prefix", repr(ID_PREFIX), "| hardmask", HARDMASK_REPEATS, "| batch", BATCH_SIZE, "| source", GENOME_SOURCE)

In [ ]:
#@title 6 · Collect genome(s) { display-mode: "form" }
import os, shutil, gzip

STAGE = "/content/run"; GEN_DIR = os.path.join(STAGE, "genomes"); os.makedirs(GEN_DIR, exist_ok=True)
FASTA_EXT = (".fa", ".fasta", ".fna", ".fa.gz", ".fasta.gz", ".fna.gz")

def base_of(p):
    b = os.path.basename(p)
    for e in (".fa.gz", ".fasta.gz", ".fna.gz", ".fa", ".fasta", ".fna"):
        if b.lower().endswith(e):
            return b[:-len(e)]
    return os.path.splitext(b)[0]

def stage_local(src):
    # return a local, uncompressed FASTA path (gunzip / copy from Drive as needed)
    b = os.path.basename(src)
    if b.endswith(".gz"):
        out = os.path.join(GEN_DIR, b[:-3])
        with gzip.open(src, "rb") as fi, open(out, "wb") as fo:
            shutil.copyfileobj(fi, fo)
        return out
    out = os.path.join(GEN_DIR, b)
    if os.path.abspath(src) != os.path.abspath(out):
        shutil.copy(src, out)
    return out

def model_softmasking(cfg):
    # read the chosen model's yaml and return its softmasking flag
    import yaml
    root = "/content/Tiberius"
    cands = [cfg, os.path.join(root, cfg),
             os.path.join(root, "model_cfg", cfg if cfg.endswith(".yaml") else cfg + ".yaml")]
    for p in cands:
        if os.path.exists(p):
            try:
                return bool(yaml.safe_load(open(p)).get("softmasking", False))
            except Exception:
                return False
    return False

def hardmask(path):
    tab = str.maketrans({c: "N" for c in "abcdefghijklmnopqrstuvwxyz"})   # lowercase (softmasked) -> N
    out = os.path.splitext(path)[0] + ".hardmasked.fasta"
    with open(path) as fi, open(out, "w") as fo:
        for line in fi:
            fo.write(line if line.startswith(">") else line.translate(tab))
    return out

raw = []
if GENOME_SOURCE == "drive_dir":
    assert MOUNT_DRIVE, "Enable MOUNT_DRIVE in cell 4 to read from Drive."
    assert os.path.isdir(GENOME_DIR), f"Not a directory: {GENOME_DIR}"
    for root, _, fnames in os.walk(GENOME_DIR):
        for fn in fnames:
            if fn.lower().endswith(FASTA_EXT):
                raw.append(os.path.join(root, fn))
    assert raw, f"No FASTA (.fa/.fasta/.fna[.gz]) found under {GENOME_DIR}"
elif GENOME_SOURCE == "drive_file":
    assert os.path.exists(GENOME_PATH), f"Not found: {GENOME_PATH}"
    raw = [GENOME_PATH]
elif GENOME_SOURCE == "url":
    assert GENOME_URL, "Set GENOME_URL in cell 5."
    dst = os.path.join(GEN_DIR, GENOME_URL.split("/")[-1])
    !wget -q -O "$dst" "$GENOME_URL"
    raw = [dst]
elif GENOME_SOURCE == "upload":
    from google.colab import files
    up = files.upload()
    for name in up:
        d = os.path.join(GEN_DIR, name); os.replace(name, d); raw.append(d)

GENOMES = [(base_of(p), stage_local(p)) for p in sorted(raw)]

if HARDMASK_REPEATS:
    if model_softmasking(MODEL_CFG):
        print("NOTE: selected model uses softmasking - skipping hardmask (it needs the lowercase mask).")
    else:
        GENOMES = [(b, hardmask(p)) for b, p in GENOMES]
        print("Hardmasked (lowercase -> N) all genomes.")

print(f"{len(GENOMES)} genome(s) staged:")
for b, p in GENOMES:
    print("  ", b, "->", p)

In [ ]:
#@title 7 · Run Tiberius on all genome(s) { display-mode: "form" }
import os, gzip, shutil, subprocess
PRED_LOCAL = "/content/run/pred"; LOG_LOCAL = "/content/run/log"
os.makedirs(PRED_LOCAL, exist_ok=True); os.makedirs(LOG_LOCAL, exist_ok=True)

OUTBASE = os.path.join(WORKDIR, "tiberius_predictions")
DIRS = {k: os.path.join(OUTBASE, k) for k in ("gff", "cds", "protein", "log")}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

def gzip_to(src, dst_dir, level=6):
    dst = os.path.join(dst_dir, os.path.basename(src) + ".gz")
    with open(src, "rb") as fi, gzip.open(dst, "wb", compresslevel=level) as fo:
        shutil.copyfileobj(fi, fo)
    return dst

def run_logged(cmd, logpath):
    # stream stdout+stderr to the console and a log file simultaneously
    with open(logpath, "w") as log:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end=""); log.write(line)
        p.wait()
    return p.returncode

%cd /content/Tiberius
JOBS = []
for i, (base, gpath) in enumerate(GENOMES, 1):
    print(f"\n===== [{i}/{len(GENOMES)}] {base} =====")
    gff  = os.path.join(PRED_LOCAL, f"{base}_gff.gff3")
    cds  = os.path.join(PRED_LOCAL, f"{base}_cds.fasta")
    prot = os.path.join(PRED_LOCAL, f"{base}_protein.fasta")
    log_local = os.path.join(LOG_LOCAL, f"{base}.log")
    cmd = ["python", "tiberius.py", "--genome", gpath, "--model_cfg", MODEL_CFG, "--out", gff]
    if ID_PREFIX:    cmd += ["--id_prefix", ID_PREFIX]
    if WANT_CODING:  cmd += ["--codingseq", cds]
    if WANT_PROTEIN: cmd += ["--protseq", prot]
    if BATCH_SIZE != "auto": cmd += ["--batch_size", str(int(BATCH_SIZE))]
    print(" ".join(cmd))
    rc = run_logged(cmd, log_local)
    shutil.copy(log_local, DIRS["log"])              # keep the log even on failure
    if rc != 0:
        raise RuntimeError(f"Tiberius failed on {base} (exit {rc}); see {os.path.join(DIRS['log'], base + '.log')}")
    saved = [gzip_to(gff, DIRS["gff"])]
    if WANT_CODING and os.path.exists(cds):   saved.append(gzip_to(cds, DIRS["cds"]))
    if WANT_PROTEIN and os.path.exists(prot): saved.append(gzip_to(prot, DIRS["protein"]))
    print("saved:", ", ".join(os.path.relpath(s, OUTBASE) for s in saved))
    JOBS.append({"base": base, "genome": gpath, "gff": gff, "cds": cds, "protein": prot})

print(f"\nDone. {len(JOBS)} prediction(s) under {OUTBASE}/  (gff/ cds/ protein/ log/)")

In [ ]:
#@title 8 · Summary of predictions { display-mode: "form" }
import os, gzip, statistics as st
from collections import Counter
import pandas as pd
from IPython.display import display

PRED_OUT = os.path.join(WORKDIR, "tiberius_predictions")

def _resolve(local, subdir):
    # prefer the local uncompressed file; fall back to the gzipped copy on Drive
    if os.path.exists(local):
        return local
    gz = os.path.join(PRED_OUT, subdir, os.path.basename(local) + ".gz")
    return gz if os.path.exists(gz) else local

def _open(p):
    return gzip.open(p, "rt") if p.endswith(".gz") else open(p)

def gff_stats(gff):
    feat = Counter(); cds_bp = 0
    with _open(gff) as fh:
        for ln in fh:
            if ln.startswith("#"):
                continue
            f = ln.rstrip("\n").split("\t")
            if len(f) < 8:
                continue
            feat[f[2]] += 1
            if f[2].lower() == "cds":
                try:
                    cds_bp += int(f[4]) - int(f[3]) + 1
                except ValueError:
                    pass
    return feat, cds_bp

def prot_lengths(fa):
    L = []; cur = 0
    if not os.path.exists(fa):
        return L
    with _open(fa) as fh:
        for ln in fh:
            if ln.startswith(">"):
                if cur:
                    L.append(cur)
                cur = 0
            else:
                cur += sum(1 for ch in ln.strip() if ch != "*")
    if cur:
        L.append(cur)
    return L

rows = []
for j in JOBS:
    feat, cds_bp = gff_stats(_resolve(j["gff"], "gff"))
    L = prot_lengths(_resolve(j["protein"], "protein"))
    mrna = feat.get("mRNA", 0) or feat.get("transcript", 0)
    rows.append({
        "genome": j["base"],
        "genes": feat.get("gene", 0),
        "mRNA": mrna,
        "proteins": len(L),
        "CDS": feat.get("CDS", 0),
        "exons": feat.get("exon", 0),
        "exons/mRNA": round(feat.get("exon", 0) / mrna, 2) if mrna else 0,
        "mean_aa": round(st.mean(L)) if L else 0,
        "median_aa": round(st.median(L)) if L else 0,
        "max_aa": max(L) if L else 0,
        "total_CDS_Mb": round(cds_bp / 1e6, 2),
    })

df = pd.DataFrame(rows)
out_tsv = os.path.join(WORKDIR, "prediction_summary.tsv")
df.to_csv(out_tsv, sep="\t", index=False)
display(df)
print("Saved:", out_tsv)
print("Outputs under:", PRED_OUT, " (gff/ cds/ protein/ log/)")

## BUSCO completeness

Two independent checks, each toggled in the next cell:
- **proteins mode** — runs on Tiberius's predicted proteins; how complete the **annotation** is. HMMER only -> fast (needs `WANT_PROTEIN = True`).
- **genome mode** — runs on each assembly via miniprot; the assembly's completeness **ceiling**, independent of annotation. Needs bbtools + miniprot and is **much slower** (CPU-bound — for whole genomes prefer HPC).

Pick a lineage from the list (plants, fungi, algae, or generic `eukaryota`), or choose **other** and type any dataset name. List all with `busco --list-datasets`. Every genome in the batch is scored against the same lineage.

In [ ]:
#@title 9 · BUSCO options { display-mode: "form" }
RUN_PROTEIN_BUSCO = True   #@param {type:"boolean"}
RUN_GENOME_BUSCO = False   #@param {type:"boolean"}
BUSCO_LINEAGE = "eudicotyledons_odb12.2"  #@param ["eukaryota_odb12.2", "viridiplantae_odb12.2", "embryophyta_odb12.2", "eudicotyledons_odb12.2", "liliopsida_odb12.2", "fungi_odb12.2", "ascomycota_odb12.2", "basidiomycota_odb12.2", "eurotiomycetes_odb12.2", "sordariomycetes_odb12.2", "saccharomycetes_odb12.2", "chlorophyta_odb12.2", "chlorophyceae_odb12.2", "trebouxiophyceae_odb12.2", "bacillariophyta_odb12.2", "stramenopiles_odb12.2", "other"] {allow-input: true}
#@markdown If `BUSCO_LINEAGE` is **other**, type the dataset name here:
BUSCO_LINEAGE_CUSTOM = ""  #@param {type:"string"}

LINEAGE = BUSCO_LINEAGE_CUSTOM.strip() if (BUSCO_LINEAGE == "other" or BUSCO_LINEAGE_CUSTOM.strip()) else BUSCO_LINEAGE
assert LINEAGE and LINEAGE != "other", "Choose a lineage, or set BUSCO_LINEAGE_CUSTOM when using 'other'."
print("lineage:", LINEAGE)

In [ ]:
#@title 10 · Install BUSCO (+ genome-mode tools if enabled) { display-mode: "form" }
#@markdown HMMER + BUSCO for both modes. Genome mode also builds miniprot and installs bbtools. ~1-3 min.
!apt-get -qq install -y hmmer > /dev/null
%cd /content
![ -d busco ] || git clone --depth 1 https://gitlab.com/ezlab/busco.git
!pip install -q ./busco biopython pandas requests scipy

if RUN_GENOME_BUSCO:
    import os
    !apt-get -qq install -y bbmap > /dev/null
    !ln -sf /usr/share/bbmap/stats.sh /usr/bin/stats.sh
    !ln -sf /usr/share/bbmap/statswrapper.sh /usr/bin/statswrapper.sh
    if not os.path.exists("/usr/local/bin/miniprot"):
        !git clone --depth 1 https://github.com/lh3/miniprot /content/miniprot
        !make -C /content/miniprot -j2 && cp /content/miniprot/miniprot /usr/local/bin/
    !echo "miniprot: $(miniprot --version)"
!busco --version

In [ ]:
#@title 11 · Run BUSCO + compile results table { display-mode: "form" }
import os, glob, re, csv, subprocess
BUSCO_LOCAL = "/content/busco_runs"; os.makedirs(BUSCO_LOCAL, exist_ok=True)
assert RUN_PROTEIN_BUSCO or RUN_GENOME_BUSCO, "Both BUSCO toggles are off (cell 9)."

def parse_summary(path):
    m = re.search(r"C:([\d.]+)%\[S:([\d.]+)%,D:([\d.]+)%\],F:([\d.]+)%,M:([\d.]+)%,n:(\d+)", open(path).read())
    keys = ["C%", "S%", "D%", "F%", "M%", "n"]
    return dict(zip(keys, m.groups())) if m else {k: "" for k in keys}

rows = []
for j in JOBS:
    targets = []
    if RUN_PROTEIN_BUSCO:
        assert os.path.exists(j["protein"]), f"No protein FASTA for {j['base']} - enable WANT_PROTEIN and re-run cell 7."
        targets.append(("proteins", j["protein"], ["-m", "proteins"], f"{j['base']}_busco_prot"))
    if RUN_GENOME_BUSCO:
        targets.append(("genome", j["genome"], ["-m", "genome", "--miniprot"], f"{j['base']}_busco_genome"))
    for mode, inp, modeargs, name in targets:
        cmd = ["busco", "-i", inp, *modeargs, "-l", LINEAGE, "-o", name,
               "--out_path", BUSCO_LOCAL, "-c", str(os.cpu_count() or 2), "-f"]
        print(f"\n##### BUSCO {mode}: {j['base']} #####")
        subprocess.run(cmd, check=True)
        summ = glob.glob(os.path.join(BUSCO_LOCAL, name, "short_summary*.txt"))
        rows.append({"genome": j["base"], "mode": mode, "lineage": LINEAGE,
                     **(parse_summary(summ[0]) if summ else {})})

cols = ["genome", "mode", "lineage", "C%", "S%", "D%", "F%", "M%", "n"]
table = os.path.join(WORKDIR, "busco_summary.tsv")
with open(table, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols, delimiter="\t"); w.writeheader(); w.writerows(rows)

print("\n=== BUSCO summary table ===")
print("\t".join(cols))
for r in rows:
    print("\t".join(str(r.get(c, "")) for c in cols))
print("\nTable saved:", table)

In [ ]:
#@title 12 · Compress & save BUSCO run directories { display-mode: "form" }
import os, glob, shutil
BUSCO_LOCAL = "/content/busco_runs"
BUSCO_OUT = os.path.join(WORKDIR, "busco_results"); os.makedirs(BUSCO_OUT, exist_ok=True)
for d in sorted(glob.glob(os.path.join(BUSCO_LOCAL, "*"))):
    if os.path.isdir(d):
        arc = shutil.make_archive(os.path.join(BUSCO_OUT, os.path.basename(d)), "zip", d)
        print("zipped:", os.path.basename(arc))
if not MOUNT_DRIVE:
    from google.colab import files
    for z in glob.glob(os.path.join(BUSCO_OUT, "*.zip")):
        files.download(z)
print("\nBUSCO zips in:", BUSCO_OUT)
print("Summary table: busco_summary.tsv in", WORKDIR)

---
### Notes & troubleshooting
- **Outputs** land in `WORKDIR`: predictions in `tiberius_predictions/` split into `gff/`, `cds/`, `protein/` (gzipped, named `<genome>_gff/_cds/_protein`) and `log/` (per-genome run log); BUSCO run zips in `busco_results/`; and the compiled `prediction_summary.tsv` (per-genome mRNA/protein/CDS stats) and `busco_summary.tsv`.
- **Results persist on Drive** when `MOUNT_DRIVE` is on; with it off, files are ephemeral and the save cells offer downloads instead.
- **Edit a hidden cell:** menu (vertical dots) > *Show code*.
- **OOM on the GPU:** set `BATCH_SIZE` to 2 or 4 in cell 5 (T4 ~ 4; 24 GB ~ 8; 80 GB ~ 16).
- **GPU not detected after install:** *Runtime > Restart session*, then re-run cells 1-3 (no reinstall).
- **BUSCO + Drive:** BUSCO uses symlinks (unsupported on the Drive mount), so it always runs in a local folder (`/content/busco_runs`); cell 12 compresses each run into Drive.
- **Genome BUSCO is slow** on Colab cores — fine for scaffolds, better on HPC for whole genomes.
- **Lineages:** `busco --list-datasets` lists every dataset; pick the most specific clade for your species.
- Cite: Gabriel *et al.* (2024) *Bioinformatics* 40(12):btae685; Manni *et al.* (2021) for BUSCO.